<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 07 · REAL-TIME ANALYTICS WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Maintain Current State and Remove Data Safely</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:920px;margin:0">Apply ordered state changes, preserve untouched columns, correct selected rows, and choose a deletion method that matches the amount of data being removed.</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">Unique Key · Upsert · Sequence column · Partial column update · UPDATE · MERGE INTO · DELETE · Delete Sign · TRUNCATE · INSERT OVERWRITE</span>
</div>

This lab uses small, independent order tables. It does not modify `events`, `events_modelled`, or `dim_products`. Each write section restores its own controlled starting state before applying a change, so you can rerun that section without accumulating rows.

Choose the operation from the change contract:

| Change contract | Doris operation |
|---|---|
| A complete incoming state for each business key | Load-based full-row upsert |
| Only selected value columns changed | Partial column update |
| A low-frequency correction selected by a predicate | SQL `UPDATE` |
| A staged relation contains conditional inserts, updates, and deletes | `MERGE INTO` |
| Remove rows selected by a predicate | SQL `DELETE` |
| A batch or CDC stream carries deleted primary keys | Delete Sign |
| Remove an entire table or Partition | `TRUNCATE` |
| Rebuild a table or Partition without an empty-data interval | `INSERT OVERWRITE` |


### Initialize the Lab

Run the next cell before Section 1. It loads the shared course helper and creates the `lab` object used by every later cell. Run it again after restarting the Jupyter kernel. It does not start Docker or change data in Doris.


In [1]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT);


## 1. Keep one current row per order as changes arrive

An event-history table preserves every event, but `order_state` answers a different question: what is the current state of each order? Its grain is one logical row per `order_id`, so a Unique Key model is appropriate.

`updated_at` is the Sequence column. When multiple writes carry the same `order_id`, the row with the larger Sequence value remains visible. The other columns are value columns that describe that state.

An upsert combines update and insert semantics in one write: when the Key already exists, Doris replaces its visible state; when the Key does not exist, Doris inserts a new logical row. After displaying three initial orders, the same cell writes a complete `shipped` state for existing order `1001` and inserts new order `1004`. A delayed `paid` event for order `1001` arrives last, but its `updated_at` value is five minutes older than the visible `shipped` state.


In [2]:
lab.connect(container="doris", host="127.0.0.1", port=9030)
lab.execute("USE doris_course")
lab.execute("DROP TABLE IF EXISTS order_state")
lab.execute("""
CREATE TABLE order_state (
    order_id BIGINT NOT NULL,
    customer_id BIGINT NOT NULL,
    status VARCHAR(20) NOT NULL DEFAULT 'created',
    amount DECIMAL(12, 2) NOT NULL DEFAULT '0.00',
    shipping_region VARCHAR(16) NOT NULL DEFAULT 'UNASSIGNED',
    updated_at DATETIME NOT NULL
)
UNIQUE KEY(order_id)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES (
    "replication_num" = "1",
    "function_column.sequence_col" = "updated_at"
)
""")

lab.insert("""
INSERT INTO order_state VALUES
    (1001, 501, 'created',    99.90, 'CN-EAST',  '2026-09-11 09:00:00'),
    (1002, 502, 'paid',      120.00, 'CN-SOUTH', '2026-09-11 09:05:00'),
    (1003, 503, 'processing', 88.00, 'CN-WEST',  '2026-09-11 09:10:00')
""", title="Write the initial order states")

lab.sql("SELECT * FROM order_state ORDER BY order_id", title="Initial current-state rows")

lab.insert("""
INSERT INTO order_state VALUES
    (1001, 501, 'shipped', 99.90, 'CN-EAST',  '2026-09-11 10:10:00'),
    (1004, 504, 'created', 42.00, 'CN-NORTH', '2026-09-11 10:00:00')
""", title="Upsert a newer state and a new order")

lab.insert("""
INSERT INTO order_state VALUES
    (1001, 501, 'paid', 99.90, 'CN-EAST', '2026-09-11 10:05:00')
""", title="Send a delayed older state")

lab.sql("""
SELECT order_id, status, amount, updated_at
FROM order_state
WHERE order_id IN (1001, 1004)
ORDER BY order_id
""", title="Visible states after ordered upserts");


order_id,customer_id,status,amount,shipping_region,updated_at
1001,501,created,99.90,CN-EAST,2026-09-11 09:00:00
1002,502,paid,120.00,CN-SOUTH,2026-09-11 09:05:00
1003,503,processing,88.00,CN-WEST,2026-09-11 09:10:00


order_id,status,amount,updated_at
1001,shipped,99.90,2026-09-11 10:10:00
1004,created,42.00,2026-09-11 10:00:00


**Expected result:** the first result contains three initial rows, one for each `order_id`. The final result shows order `1001` as `shipped` at `10:10`, even though the `paid` row arrived later, because `10:05` has a smaller Sequence value. Order `1004` appears as a new row, and the table contains four logical rows rather than six historical versions.

This result demonstrates both branches of upsert: `1001` updates an existing Key and `1004` inserts a new Key. In Doris 4.x, the Unique Key model uses Merge-on-Write by default. According to the documented mechanism, Doris writes an accepted replacement into a new Rowset and records the replaced row in a delete bitmap; subsequent queries skip that old row version, and physical cleanup happens later during Compaction. The visible rows above demonstrate the winning state, not those internal structures directly. Re-running this cell drops and rebuilds the table before applying the same changes.


## 2. Choose whether omitted columns use defaults or retain existing values

Both test orders begin with an amount and shipping region. The full-row upsert for order `1002` omits those columns, so Doris fills them from their declared defaults. The partial column update for order `1003` changes only `status` and `updated_at`; Doris retains the existing values of the omitted columns.

For `INSERT INTO`, the session variable `enable_unique_key_partial_update` makes that distinction explicit. Stream Load and other ingestion paths use their corresponding `partial_columns` setting.


In [3]:
lab.execute("TRUNCATE TABLE order_state")
lab.insert("""
INSERT INTO order_state VALUES
    (1002, 502, 'paid', 120.00, 'CN-SOUTH', '2026-09-11 09:05:00'),
    (1003, 503, 'paid',  88.00, 'CN-WEST',  '2026-09-11 09:10:00')
""", title="Restore two complete order states")

lab.execute("SET enable_unique_key_partial_update = false")
lab.insert("""
INSERT INTO order_state (order_id, customer_id, status, updated_at)
VALUES (1002, 502, 'shipped', '2026-09-11 10:20:00')
""", title="Apply a full-row upsert with omitted columns")

lab.execute("SET enable_unique_key_partial_update = true")
lab.insert("""
INSERT INTO order_state (order_id, status, updated_at)
VALUES (1003, 'shipped', '2026-09-11 10:20:00')
""", title="Update only the changed columns")
lab.execute("SET enable_unique_key_partial_update = false")

lab.sql("""
SELECT
    order_id, status, amount, shipping_region,
    IF(order_id = 1002, 'full-row upsert', 'partial column update') AS update_path
FROM order_state
ORDER BY order_id
""", title="Full-row and partial-update results");


order_id,status,amount,shipping_region,update_path
1002,shipped,0.00,UNASSIGNED,full-row upsert
1003,shipped,88.00,CN-WEST,partial column update


**Expected result:** both orders become `shipped`. Order `1002` now has `amount = 0.00` and `shipping_region = UNASSIGNED`, because the full-row upsert used the column defaults. Order `1003` retains `88.00` and `CN-WEST`, because the partial column update preserved the values it did not receive. Use partial updates when the producer owns only part of a wide current-state row.


## 3. Correct a small selected set with SQL UPDATE

SQL `UPDATE` is useful for a low-frequency correction expressed by a predicate. Doris scans the rows selected by `WHERE` and writes updated row versions back to the Unique Key table. It is not the preferred path for a tight loop of individual application writes.

Only value columns can be changed with `UPDATE`. Changing `order_id` would change the business Key and must instead be represented as deleting the old Key and inserting the new Key.

| Update path | How rows are selected | Suitable starting point | Untouched value columns |
|---|---|---|---|
| Full-row upsert | Incoming Key values | Batched complete states or CDC records | Replaced by supplied values or declared defaults |
| Partial column update | Incoming Key values | Batched changes that contain only selected columns | Preserved from the existing row |
| SQL `UPDATE` | SQL `WHERE` predicate | Occasional conditional correction | Preserved unless named in `SET` |


In [4]:
lab.execute("TRUNCATE TABLE order_state")
lab.insert("""
INSERT INTO order_state VALUES
    (1001, 501, 'shipped', 99.90, 'CN-EAST', '2026-09-11 10:10:00'),
    (1002, 502, 'paid',   120.00, 'CN-SOUTH', '2026-09-11 09:05:00')
""", title="Restore the rows used by the correction")

lab.sql("""
SELECT order_id, status, amount, shipping_region, updated_at
FROM order_state
ORDER BY order_id
""", title="Order states before SQL UPDATE")

lab.insert("""
UPDATE order_state
SET shipping_region = 'CN-NORTH'
WHERE order_id = 1001
""", title="Correct one shipping region")

lab.sql("""
SELECT order_id, status, amount, shipping_region, updated_at
FROM order_state
ORDER BY order_id
""", title="Order states after the correction");


order_id,status,amount,shipping_region,updated_at
1001,shipped,99.90,CN-EAST,2026-09-11 10:10:00
1002,paid,120.00,CN-SOUTH,2026-09-11 09:05:00


order_id,status,amount,shipping_region,updated_at
1001,shipped,99.90,CN-NORTH,2026-09-11 10:10:00
1002,paid,120.00,CN-SOUTH,2026-09-11 09:05:00


**Expected result:** compare the two result tables. Before `UPDATE`, order `1001` has `shipping_region = CN-EAST`; afterward it has `CN-NORTH`. Its status, amount, and timestamp remain unchanged, and every value for order `1002` is unaffected. The `WHERE` predicate selected the row, while `SET` named the only value column to change.

Together, Sections 1–3 distinguish the three paths: full-row upsert writes complete incoming states, partial column update preserves omitted values, and SQL `UPDATE` performs an occasional predicate-based correction. High-volume change streams should normally use batched load-based upserts rather than many individual SQL `UPDATE` statements.


## 4. Apply conditional changes from a staged relation with MERGE INTO

A local staging table contains three source changes: update existing order `4001`, delete existing order `4002`, and insert new order `4004`. `MERGE INTO` fits because one source relation requires different actions according to match status and `change_type`.

The target must be a Unique Key table. Before executing the merge, compare source row count with distinct `order_id` count. Doris does not detect duplicate Join rows in `MERGE INTO`; several source rows acting on the same target row can produce undefined behavior. A production pipeline should deduplicate by an authoritative source sequence or reject the batch.


In [5]:
lab.execute("DROP TABLE IF EXISTS order_merge_changes")
lab.execute("DROP TABLE IF EXISTS order_merge_target")
lab.execute("""
CREATE TABLE order_merge_target (
    order_id BIGINT NOT NULL,
    status VARCHAR(20) NOT NULL,
    amount DECIMAL(12, 2) NOT NULL,
    shipping_region VARCHAR(16) NOT NULL,
    updated_at DATETIME NOT NULL
)
UNIQUE KEY(order_id)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")
lab.execute("""
CREATE TABLE order_merge_changes (
    order_id BIGINT NOT NULL,
    change_type VARCHAR(10) NOT NULL,
    status VARCHAR(20) NULL,
    amount DECIMAL(12, 2) NULL,
    shipping_region VARCHAR(16) NULL,
    updated_at DATETIME NULL
)
DUPLICATE KEY(order_id, change_type)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")
lab.insert("""
INSERT INTO order_merge_target VALUES
    (4001, 'paid',       99.90, 'CN-EAST',  '2026-09-11 09:00:00'),
    (4002, 'processing',120.00, 'CN-SOUTH', '2026-09-11 09:05:00'),
    (4003, 'paid',       75.00, 'CN-WEST',  '2026-09-11 09:10:00')
""", title="Write the MERGE target state")
lab.insert("""
INSERT INTO order_merge_changes VALUES
    (4001, 'UPSERT', 'shipped', 99.90, 'CN-EAST',  '2026-09-11 10:00:00'),
    (4002, 'DELETE', NULL,      NULL,  NULL,       NULL),
    (4004, 'UPSERT', 'created', 50.00, 'CN-NORTH', '2026-09-11 10:05:00')
""", title="Stage update, delete, and insert changes")
lab.sql("""
SELECT
    COUNT(*) AS source_rows,
    COUNT(DISTINCT order_id) AS distinct_order_keys,
    IF(COUNT(*) = COUNT(DISTINCT order_id), 'ready', 'reject') AS merge_readiness
FROM order_merge_changes
""", title="Validate one source row per target Key")
lab.insert("""
MERGE INTO order_merge_target AS t
USING order_merge_changes AS s
ON t.order_id = s.order_id
WHEN MATCHED AND s.change_type = 'DELETE' THEN DELETE
WHEN MATCHED AND s.change_type = 'UPSERT' THEN UPDATE SET
    status = s.status,
    amount = s.amount,
    shipping_region = s.shipping_region,
    updated_at = s.updated_at
WHEN NOT MATCHED AND s.change_type = 'UPSERT' THEN INSERT
    (order_id, status, amount, shipping_region, updated_at)
VALUES
    (s.order_id, s.status, s.amount, s.shipping_region, s.updated_at)
""", title="Merge the staged change set")
lab.sql("SELECT * FROM order_merge_target ORDER BY order_id", title="Current state after MERGE INTO");


source_rows,distinct_order_keys,merge_readiness
3,3,ready


order_id,status,amount,shipping_region,updated_at
4001,shipped,99.90,CN-EAST,2026-09-11 10:00:00
4003,paid,75.00,CN-WEST,2026-09-11 09:10:00
4004,created,50.00,CN-NORTH,2026-09-11 10:05:00


**Expected result:** the validation row reports `source_rows = 3`, `distinct_order_keys = 3`, and `merge_readiness = ready`. After the merge:

- `4001` is `shipped` at `10:00` because a matched UPSERT selected the `UPDATE` action;
- `4002` is absent because a matched DELETE selected the `DELETE` action;
- `4003` remains unchanged because no source row matched it; and
- `4004` is `created` at `10:05` because a not-matched UPSERT selected the `INSERT` action.

The final target therefore contains three visible rows: `4001`, `4003`, and `4004`. This proves the actions and result for this unique three-row source. It does not establish behavior for duplicate source matches, which the source contract deliberately rejects.


## 5. Remove rows by predicate or incoming Key

The table contains one retained row and two rows to remove. SQL `DELETE` selects a test row with a predicate. The second removal writes `__DORIS_DELETE_SIGN__ = 1` for an incoming Key, which models a batched delete or a delete event from a CDC source.

Ordinary queries hide deleted rows. Enabling `show_hidden_columns` makes the delete marker visible for this inspection; application queries normally leave it disabled. `INSERT INTO` is used here to expose the mechanism with one line. Production CDC and large batches normally carry the same marker through a load path.

The two Tablet snapshots provide storage evidence around the deletions. `Version` is the latest published Tablet version, while `VersionCount` reports the versions currently retained by the replica. `RowCount` is storage metadata and must not be interpreted as the number of rows visible to an ordinary query.


In [6]:
lab.execute("DROP TABLE IF EXISTS order_deletions")
lab.execute("""
CREATE TABLE order_deletions (
    order_id BIGINT NOT NULL,
    status VARCHAR(20) NOT NULL DEFAULT 'unknown',
    updated_at DATETIME NOT NULL DEFAULT '1970-01-01 00:00:00'
)
UNIQUE KEY(order_id)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")
lab.insert("""
INSERT INTO order_deletions VALUES
    (2001, 'paid',      '2026-09-11 09:00:00'),
    (2002, 'cancelled', '2026-09-11 09:05:00'),
    (2003, 'test',      '2026-09-11 09:10:00')
""", title="Write the controlled deletion cases")

lab.sql("""
SELECT order_id, status, updated_at, COUNT(*) OVER() AS visible_row_count
FROM order_deletions
ORDER BY order_id
""", title="Visible rows before deletion")
lab.sql(
    "SHOW TABLETS FROM order_deletions",
    title="Tablet version before deletion",
    columns=["TabletId", "Version", "VersionCount", "RowCount"],
)

lab.execute("DELETE FROM order_deletions WHERE order_id = 2003")
lab.execute("SET show_hidden_columns = true")
lab.insert("""
INSERT INTO order_deletions (order_id, __DORIS_DELETE_SIGN__)
VALUES (2002, 1)
""", title="Write an incoming Delete Sign")
lab.execute("SET show_hidden_columns = false")

lab.sql("""
SELECT order_id, status, updated_at, COUNT(*) OVER() AS visible_row_count
FROM order_deletions
ORDER BY order_id
""", title="Visible rows after deletion")
lab.sql(
    "SHOW TABLETS FROM order_deletions",
    title="Tablet version after deletion",
    columns=["TabletId", "Version", "VersionCount", "RowCount"],
)

lab.execute("SET show_hidden_columns = true")
lab.sql("""
SELECT order_id, status, updated_at, __DORIS_DELETE_SIGN__
FROM order_deletions
ORDER BY order_id
""", title="Stored delete markers")
lab.execute("SET show_hidden_columns = false");


order_id,status,updated_at,visible_row_count
2001,paid,2026-09-11 09:00:00,3
2002,cancelled,2026-09-11 09:05:00,3
2003,test,2026-09-11 09:10:00,3


TabletId,Version,VersionCount,RowCount
1789376896229,2,-1,0


order_id,status,updated_at,visible_row_count
2001,paid,2026-09-11 09:00:00,1


TabletId,Version,VersionCount,RowCount
1789376896229,4,-1,0


order_id,status,updated_at,__DORIS_DELETE_SIGN__
2001,paid,2026-09-11 09:00:00,0
2002,unknown,1970-01-01 00:00:00,1
2003,unknown,1970-01-01 00:00:00,1


**Expected result:** before deletion, three rows are visible and each row reports `visible_row_count = 3`; their `status` and `updated_at` values show the starting state. After the two deletion operations, the ordinary query returns only order `2001`, with `status = paid`, `updated_at = 2026-09-11 09:00:00`, and `visible_row_count = 1`. The second Tablet snapshot reports a later published `Version` than the first snapshot. `VersionCount` may increase or may already have been reduced by background Compaction, so its exact value is not graded.

The ordinary query result and the later Tablet `Version` are directly observed evidence: the transaction changed logical visibility and published new storage versions. According to the Doris Merge-on-Write mechanism, old row versions are marked as invisible in a delete bitmap and are cleaned by later Compaction. The changed query result therefore represents logical deletion; it does not mean that physical storage is reclaimed immediately.

With hidden columns enabled, orders `2002` and `2003` have `__DORIS_DELETE_SIGN__ = 1`: `2002` received the marker directly, while the predicate `DELETE` made `2003` logically deleted. A delete record may contain defaults rather than the previous business values, so use this view to inspect the marker—not to reconstruct deleted state. Delete Sign is available only for Unique Key tables and is useful when many deleted Keys already arrive as input data. Predicate `DELETE` is the general choice when a SQL condition defines the rows to remove. `__DORIS_DELETE_SIGN__` is an input-facing hidden column; it is not the internal per-Rowset delete bitmap.


## 6. Match whole-Partition maintenance to a lifecycle operation

Deleting every row in a Partition one row at a time performs unnecessary work. The next table separates two order dates into Partitions. `TRUNCATE PARTITION` removes the expired date through metadata, while `INSERT OVERWRITE` builds replacement contents and swaps them into the selected Partition atomically.

The queries show the state after each lifecycle operation. These operations are run only on the small `order_lifecycle` table.


In [7]:
lab.execute("DROP TABLE IF EXISTS order_lifecycle")
lab.execute("""
CREATE TABLE order_lifecycle (
    order_date DATE NOT NULL,
    order_id BIGINT NOT NULL,
    status VARCHAR(20) NOT NULL
)
DUPLICATE KEY(order_date, order_id)
PARTITION BY RANGE(order_date) (
    PARTITION p20260910 VALUES LESS THAN ('2026-09-11'),
    PARTITION p20260911 VALUES LESS THAN ('2026-09-12'),
    PARTITION pmax VALUES LESS THAN MAXVALUE
)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")
lab.insert("""
INSERT INTO order_lifecycle VALUES
    ('2026-09-10', 3001, 'expired'),
    ('2026-09-10', 3002, 'expired'),
    ('2026-09-11', 3003, 'paid'),
    ('2026-09-11', 3004, 'paid')
""", title="Write two daily Partitions")

lab.sql("""
SELECT order_date, COUNT(*) AS order_rows
FROM order_lifecycle
GROUP BY order_date
ORDER BY order_date
""", title="Rows before lifecycle maintenance")

lab.execute("TRUNCATE TABLE order_lifecycle PARTITION(p20260910)")
lab.sql("""
SELECT order_date, COUNT(*) AS order_rows
FROM order_lifecycle
GROUP BY order_date
ORDER BY order_date
""", title="Rows after TRUNCATE PARTITION")

lab.insert("""
INSERT OVERWRITE TABLE order_lifecycle PARTITION(p20260911) VALUES
    ('2026-09-11', 3003, 'shipped'),
    ('2026-09-11', 3005, 'created')
""", title="Atomically replace the active Partition")
lab.sql("SELECT * FROM order_lifecycle ORDER BY order_id", title="Rows after INSERT OVERWRITE");


order_date,order_rows
2026-09-10,2
2026-09-11,2


order_date,order_rows
2026-09-11,2


order_date,order_id,status
2026-09-11,3003,shipped
2026-09-11,3005,created


**Expected result:** the first result contains two rows for each date. After `TRUNCATE PARTITION`, only the two `2026-09-11` rows remain. After `INSERT OVERWRITE`, that Partition contains order `3003` with corrected status `shipped` and new order `3005`; old order `3004` is no longer present. Readers can continue to see the old Partition contents until the overwrite transaction commits, avoiding a delete-then-reload gap.


### Stop the Doris sandbox

Run this optional cell to release CPU and memory. Docker named volumes preserve the Module 7 tables.


In [8]:
lab.shell(r"""
set -euo pipefail

docker stop doris
docker inspect --format 'container={{.State.Status}}' doris
""", title="Stop the Doris sandbox");


doris


container=exited


**Expected result:** Docker reports `container=exited`.

### Restart the Doris sandbox

Run this cell before continuing to another module. It starts the existing container, reconnects to FE, and verifies the persisted current-state table.

This restart cell is idempotent: it can be run when Docker Desktop and the container are stopped, starting, or already running. On macOS it opens Docker Desktop when necessary; on Linux, start Docker Engine before running the cell.


In [ ]:
lab.start_container("doris")

lab.connect(container="doris", host="127.0.0.1", port=9030)
lab.execute("USE doris_course")
lab.sql("SELECT COUNT(*) AS current_orders FROM order_state", title="Recovered current-state table");


**Expected result:** the container returns to `healthy`, and `current_orders` reflects the two rows restored in Section 3.

## Lab complete

You modeled one current row per order, used a Sequence column to reject an older state, distinguished full-row and partial-column semantics, applied a predicate correction, merged a staged update/delete/insert set, compared predicate deletion with Delete Sign, and matched whole-Partition maintenance to `TRUNCATE` and `INSERT OVERWRITE`.

Official references: [Unique Key](https://doris.apache.org/docs/4.x/key-features/unique-key/) · [Load-Based Updates for the Unique Model](https://doris.apache.org/docs/4.x/data-operate/update/update-of-unique-model/) · [UPDATE](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/data-modification/DML/UPDATE/) · [MERGE INTO](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/data-modification/DML/MERGE-INTO/) · [Load-Based Batch Delete](https://doris.apache.org/docs/4.x/data-operate/delete/batch-delete-manual/) · [Delete Operations Overview](https://doris.apache.org/docs/4.x/data-operate/delete/delete-overview.html/) · [TRUNCATE](https://doris.apache.org/docs/4.x/data-operate/delete/truncate-manual/) · [INSERT OVERWRITE](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/data-modification/DML/INSERT-OVERWRITE/)
